In [5]:
# 필수 라이브러리 설치
!pip install requests beautifulsoup4

import re
import requests
from bs4 import BeautifulSoup
from typing import Dict, Any

# 공통 헤더 설정 (브라우저 환경 모사)
COMMON_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8",
}


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import re
import requests
from bs4 import BeautifulSoup
from typing import Dict, Any

def scrape_wanted_full_jd(url: str) -> Dict[str, Any]:
    """
    원티드 공고의 상단 소개부터 하단 채용 전형, 서류 작성 팁, 태그까지
    누락 없이 100% 수집하는 스크래퍼
    """
    match = re.search(r"/wd/(\d+)", url)
    if not match:
        raise ValueError("올바른 원티드 공고 URL이 아닙니다.")
    
    job_id = match.group(1)
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/124.0.0.0 Safari/537.36"
        ),
        "Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8",
    }
    
    # 1. REST API 호출: 기본 5대 항목 및 태그, 주소 수집
    api_url = f"https://www.wanted.co.kr/api/v4/jobs/{job_id}"
    res_api = requests.get(api_url, headers=headers, timeout=10)
    if res_api.status_code != 200:
        raise RuntimeError(f"원티드 API 호출 실패: {res_api.status_code}")
    
    job_data = res_api.json().get("job", {})
    detail = job_data.get("detail", {})
    company = job_data.get("company", {})

    company_name = company.get("name", "회사명 미상")
    position = job_data.get("position", "포지션명 미상")
    
    # 태그 및 근무지 추출
    skills = [tag.get("title") for tag in job_data.get("skill_tags", []) if tag.get("title")]
    company_tags = [tag.get("title") for tag in job_data.get("company_tags", []) if tag.get("title")]
    address = job_data.get("address", {}).get("full_location", "")

    # 2. 웹 페이지(SSR HTML) 조회: 하단 '채용 전형' 및 '서류 작성 팁' 추출
    web_url = f"https://www.wanted.co.kr/wd/{job_id}"
    res_web = requests.get(web_url, headers=headers, timeout=10)
    
    extra_guide = ""
    if res_web.status_code == 200:
        soup = BeautifulSoup(res_web.text, "html.parser")
        
        # 불필요한 공통 네비게이션 및 스크립트 제거
        for tag in soup.select("header, footer, nav, script, style, .gnb, button"):
            tag.decompose()
            
        full_page_text = soup.get_text(separator="\n", strip=True)
        
        # '채용 전형' 또는 '꼭 확인해 주세요' 시작 지점부터 텍스트 슬라이싱
        start_keywords = ["채용 전형", "꼭 확인해 주세요", "서류 작성 팁"]
        start_pos = -1
        for kw in start_keywords:
            pos = full_page_text.find(kw)
            if pos != -1:
                start_pos = pos
                break
                
        if start_pos != -1:
            # 추천 공고, 푸터 직전까지만 추출
            end_keywords = ["이 포지션을 찾고 계셨나요", "비슷한 포지션", "원티드랩", "©"]
            cut_text = full_page_text[start_pos:]
            end_pos = len(cut_text)
            for end_kw in end_keywords:
                e_pos = cut_text.find(end_kw)
                if e_pos != -1 and e_pos < end_pos:
                    end_pos = e_pos
            extra_guide = cut_text[:end_pos].strip()

    # 3. 통합 마크다운 문서 생성
    full_text = f"""# [{company_name}] {position}

## 1. 회사 및 팀 소개
{detail.get('intro', '')}

## 2. 주요 업무
{detail.get('main_tasks', '')}

## 3. 자격 요건
{detail.get('requirements', '')}

## 4. 우대 사항
{detail.get('preferred_points', '')}

## 5. 혜택 및 복지
{detail.get('benefits', '')}

## 6. 채용 전형 및 서류 작성 팁
{extra_guide if extra_guide else '별도 상세 안내 없음'}

## 7. 기업 및 기술 태그
• 요구/관련 스킬: {', '.join(skills) if skills else '정보 없음'}
• 기업 복지/문화 태그: {', '.join(company_tags) if company_tags else '정보 없음'}
• 근무 지역: {address if address else '정보 없음'}
""".strip()

    return {
        "platform": "Wanted",
        "company_name": company_name,
        "position": position,
        "full_text": full_text
    }


# 실행 테스트
wanted_test_url = "https://www.wanted.co.kr/wd/375823"
res = scrape_wanted_full_jd(wanted_test_url)

print(f"=== [원티드 전체 추출 완료: {res['company_name']}] ===")
print(res["full_text"])

=== [원티드 전체 추출 완료: 몬드리안에이아이] ===
# [몬드리안에이아이] AI/AX Engineer (신입)

## 1. 회사 및 팀 소개
[Beyond Digital Transformation, We Accelerate AI Transformation] 몬드리안에이아이는 디지털 전환을 넘어, 인공지능 전환을 가속화합니다.

세상의 모든 산업은 AI로 재정의되고 있지만, 복잡한 인프라와 높은 비용은 기업들에게 여전히 높은 진입장벽입니다.
몬드리안에이아이는 데이터 기술로 세상의 난제를 해결하고, 기업과 사회의 진정한 AI 전환(AX)을 이끄는 AI 클라우드 및 플랫폼 전문 기업입니다.

우리는 매년 150%의 가파른 매출 성장을 기록하고 있으며, 2025년 매출 50억 원 돌파 및 흑자 전환에 성공했습니다.
이 폭발적인 성장세를 바탕으로 단순한 기술 기업을 넘어, 인프라부터 서비스까지 아우르는 ‘Neo Cloud Group’으로 도약하고 있습니다.

?회사 홈페이지: https://mondrian.ai/

[? 우리의 제품과 서비스]
몬드리안에이아이는 인프라부터 플랫폼, 클라우드까지 수직 계열화(Vertical E2E)된 AI 솔루션을 제공합니다.

• 예니퍼(Yennefer): 데이터 구축부터 배포까지, AI 개발 전주기를 혁신하는 엔터프라이즈 MLOps 플랫폼
• 런유어에이아이(RunyourAI): 전 세계 유휴 자원을 연결해 고성능 컴퓨팅을 합리적으로 제공하는 GPU 공유 클라우드
• MonBox: 소프트웨어와 하드웨어가 결합된 최적의 연구 환경, AI 일체형 워크스테이션
• MonPlant: 제조 현장의 데이터를 분석해 생산성을 극대화하는 산업 특화 AI 솔루션


KAIST 연구진의 기술력으로 시작해,
이제는 삼성·SK·Dell 등 글로벌 기업들이 신뢰하는 파트너가 되었습니다.

한계 없는 기술로 AI의 미래를 설계하는 여정,
그 중심에서 함께할 동료를 기다립니다.
끝없는 성장과 일상의 혁신을 원한다면, 지금 합류하세요!



팀 소개
합류하게

In [9]:
def scrape_saramin_jd(url: str) -> Dict[str, Any]:
    """
    사람인 상세 본문(view-detail) 직호출 스크래퍼
    """
    # 1. URL에서 rec_idx 파라미터 추출
    match = re.search(r"rec_idx=(\d+)", url)
    if not match:
        raise ValueError("사람인 URL에서 rec_idx 값을 찾을 수 없습니다.")
    
    rec_idx = match.group(1)
    
    headers = COMMON_HEADERS.copy()
    headers["Referer"] = f"https://www.saramin.co.kr/zf_user/jobs/relay/view?rec_idx={rec_idx}"
    
    # 2. 메인 페이지에서 [회사명]과 [공고명] 추출
    main_url = f"https://www.saramin.co.kr/zf_user/jobs/relay/view?rec_idx={rec_idx}"
    res_main = requests.get(main_url, headers=headers, timeout=10)
    
    company_name = "회사명 미상"
    position = "공고명 미상"
    
    if res_main.status_code == 200:
        soup_main = BeautifulSoup(res_main.text, "html.parser")
        og_title = soup_main.select_one("meta[property='og:title']")
        if og_title and og_title.get("content"):
            title_text = og_title["content"].replace(" - 사람인", "").strip()
            title_match = re.match(r"^\[(.*?)\]\s*(.*)$", title_text)
            if title_match:
                company_name = title_match.group(1).strip()
                position = title_match.group(2).strip()
            else:
                position = title_text

    # 3. [핵심] 실제 본문이 들어있는 'view-detail' 엔드포인트 직접 호출
    detail_url = f"https://www.saramin.co.kr/zf_user/jobs/relay/view-detail?rec_idx={rec_idx}"
    res_detail = requests.get(detail_url, headers=headers, timeout=10)
    
    raw_body = ""
    if res_detail.status_code == 200:
        soup_detail = BeautifulSoup(res_detail.text, "html.parser")
        # 잡음 태그(스크립트, 스타일) 제거
        for tag in soup_detail.select("script, style, iframe"):
            tag.decompose()
        raw_body = soup_detail.get_text(separator="\n", strip=True)
    
    # 4. view-detail 실패 시 모바일 뷰 폴백(Fallback)
    if len(raw_body) < 100:
        mobile_url = f"https://m.saramin.co.kr/job-search/job-detail?rec_idx={rec_idx}"
        res_mobile = requests.get(mobile_url, headers=headers, timeout=10)
        if res_mobile.status_code == 200:
            soup_m = BeautifulSoup(res_mobile.text, "html.parser")
            for tag in soup_m.select("script, style, header, footer, nav, .footer, .header"):
                tag.decompose()
            content_m = soup_m.select_one(".detail_area") or soup_m.select_one(".job_detail")
            if content_m:
                raw_body = content_m.get_text(separator="\n", strip=True)

    # 5. 최종 마크다운 통합
    full_text = f"""# [{company_name}] {position}

{raw_body}
""".strip()

    return {
        "platform": "Saramin",
        "company_name": company_name,
        "position": position,
        "full_text": full_text
    }


# 실행 테스트
saramin_test_url = "https://www.saramin.co.kr/zf_user/jobs/relay/view?rec_idx=54940771"
saramin_result = scrape_saramin_jd(saramin_test_url)

print(f"=== [사람인 결과] ===")
print(f"회사명: {saramin_result['company_name']}")
print(f"공고명: {saramin_result['position']}")
print("\n--- 본문 내용 (상위 500자) ---")
print(saramin_result["full_text"][:500])
print("\n...\n(전체 추출 글자 수:", len(saramin_result["full_text"]), "자)")

=== [사람인 결과] ===
회사명: 파이특허법률사무소
공고명: [채용] 파이특허법률사무소 AI Transformation Team Intern(D-28)

--- 본문 내용 (상위 500자) ---
# [파이특허법률사무소] [채용] 파이특허법률사무소 AI Transformation Team Intern(D-28)

채용공고 상세
[채용] 파이특허법률사무소 AI Transformation Team Intern
AX TF Team Intern
(0명)
📋 주요업무
파이특허는 국내에서 가장 빠르게 성장하고 있는 특허사무소 중 하나입니다. 사내 AI 전환을 주도하는 AI Transformation(AX) TF Team에서 인턴을 찾고 있습니다.
■ 부서소개
파이특허 AI Transformation(AX) TF Team은 조직의 AI 전환을 주도하는 팀입니다. AI 관련 최신 트렌드와 기술 표준을 조사하고 실무에 적용하는 과정을 통해 회사에 맞는 AI 활용 방식을 정립하여 구성원들이 반복 업무 대신 판단이 필요한 일에 집중할 수 있도록 만드는 것이 TF 팀의 목표입니다.
■ 이런 일을 하게 됩니다
AI 관련 새로운 개념·기술을 리서치하고 실제 시스템에 반영될 수 있도록 구현합니다. 구성

...
(전체 추출 글자 수: 1608 자)
